In [0]:
# MAGIC %md
# ============================================================
# NOTEBOOK   : silver_geolocation
# PURPOSE    : Clean and deduplicate bronze.geolocation table
# SOURCE     : fincompliance_catalog.bronze.geolocation
# DESTINATION: fincompliance_catalog.silver.geolocation
# TRANSFORMATIONS:
#   - Trim and lowercase city names
#   - Uppercase state codes
#   - Reduce 1,000,163 rows to unique zip codes
#     using groupBy and average lat/lng
#   - Add silver_updated_at audit column
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper,
    avg,
    count,
    current_timestamp,
    round as spark_round
)

In [0]:
# ============================================================
# CELL 5 - READ FROM BRONZE
# ============================================================

df_geolocation = spark.table(f"{BRONZE_DB}.geolocation")

print("=" * 60)
print("BRONZE.GEOLOCATION — Before Transformation")
print("=" * 60)
print(f"Total rows    : {df_geolocation.count():,}")
print(f"Total columns : {len(df_geolocation.columns)}")

print("\nColumn names and data types:")
for field in df_geolocation.schema.fields:
    print(f"  {field.name:<35} : {field.dataType}")

print("\nSample data BEFORE transformation (5 rows):")
df_geolocation.show(5, truncate=True)

# Show why we need deduplication
print("\nUnique zip codes vs total rows:")
unique_zips = df_geolocation \
    .select("geolocation_zip_code_prefix") \
    .distinct() \
    .count()
total_rows = df_geolocation.count()
print(f"Total rows         : {total_rows:,}")
print(f"Unique zip codes   : {unique_zips:,}")
print(f"Duplication factor : {total_rows/unique_zips:.1f}x")

# Show example of same zip code with different coordinates
print("\nExample - same zip code multiple entries:")
df_geolocation \
    .filter(col("geolocation_zip_code_prefix") == 1001) \
    .show(10, truncate=False)

In [0]:
# ============================================================
# STANDARDISE CITY AND STATE NAMES
# Fix accent marks and casing issues
# "são paulo" and "sao paulo" should be the same
# ============================================================

print("Cities with accent marks (before fix):")
df_geolocation \
    .filter(col("geolocation_city").contains("ã")) \
    .select("geolocation_city") \
    .distinct() \
    .show(10, truncate=False)

# Standardise city and state
df_geolocation_clean = df_geolocation \
    .withColumn(
        "geolocation_city",
        trim(lower(col("geolocation_city")))
    ) \
    .withColumn(
        "geolocation_state",
        upper(trim(col("geolocation_state")))
    )

print("Done - City and state standardised")
print(f"Total rows still : {df_geolocation_clean.count():,}")

In [0]:
# ============================================================
# REDUCE TO UNIQUE ZIP CODES USING GROUPBY
# This is the important transformation
# in the entire project
# 1,000,163 rows → ~19,015 unique zip codes
# Average lat/lng for each zip code
# ============================================================

print("BEFORE groupBy:")
print(f"Total rows : {df_geolocation_clean.count():,}")

# Group by zip code, city, state
# Average the lat/lng coordinates
df_geolocation_silver = df_geolocation_clean \
    .groupBy(
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    ) \
    .agg(
        spark_round(avg("geolocation_lat"), 6)
            .alias("geolocation_lat"),
        spark_round(avg("geolocation_lng"), 6)
            .alias("geolocation_lng"),
        count("*").alias("entry_count")
    )

print("\nAFTER groupBy:")
print(f"Total rows : {df_geolocation_silver.count():,}")

# Show the reduction
before = df_geolocation_clean.count()
after = df_geolocation_silver.count()
reduction_pct = (1 - after/before) * 100
print(f"\nReduction: {before:,} → {after:,} rows")
print(f"Reduction percentage: {reduction_pct:.1f}%")

# Show sample of result
print("\nSample of final deduplicated data:")
df_geolocation_silver \
    .filter(col("geolocation_zip_code_prefix") == 1001) \
    .show(truncate=False)

# Show entry_count distribution
print("\nEntry count distribution (how many raw rows per zip):")
df_geolocation_silver \
    .select("entry_count") \
    .summary("min", "max", "mean") \
    .show()

In [0]:
# ============================================================
# REDUCE TO UNIQUE ZIP CODES - CORRECTED VERSION
# Group by zip_code ONLY
# City names are inconsistent (accent marks)
# Use first() to pick one representative city/state
# This avoids duplicate groups from spelling variations
# ============================================================

from pyspark.sql.functions import first

print("BEFORE groupBy:")
print(f"Total rows : {df_geolocation_clean.count():,}")

# Group by zip code ONLY
df_geolocation_silver = df_geolocation_clean \
    .groupBy("geolocation_zip_code_prefix") \
    .agg(
        spark_round(avg("geolocation_lat"), 6)
            .alias("geolocation_lat"),
        spark_round(avg("geolocation_lng"), 6)
            .alias("geolocation_lng"),
        first("geolocation_city").alias("geolocation_city"),
        first("geolocation_state").alias("geolocation_state"),
        count("*").alias("entry_count")
    )

print("\nAFTER groupBy:")
print(f"Total rows : {df_geolocation_silver.count():,}")

before = df_geolocation_clean.count()
after = df_geolocation_silver.count()
reduction_pct = (1 - after/before) * 100
print(f"\nReduction: {before:,} → {after:,} rows")
print(f"Reduction percentage: {reduction_pct:.1f}%")

# Verify zip 1001 now has only ONE row
print("\nVerify zip 1001 now has ONE row:")
df_geolocation_silver \
    .filter(col("geolocation_zip_code_prefix") == 1001) \
    .show(truncate=False)

print("\nEntry count distribution:")
df_geolocation_silver \
    .select("entry_count") \
    .summary("min", "max", "mean") \
    .show()

In [0]:
# ============================================================
# ADD SILVER AUDIT COLUMN
# ============================================================

df_geolocation_silver = df_geolocation_silver \
    .withColumn("silver_updated_at", current_timestamp())

print("Final columns:")
for col_name in df_geolocation_silver.columns:
    print(f"  {col_name}")
print(f"Total rows : {df_geolocation_silver.count():,}")

In [0]:
# ============================================================
# WRITE TO SILVER
# ============================================================

(
    df_geolocation_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.geolocation")
)

print(f"Written to {SILVER_DB}.geolocation")
print(f"Rows : {df_geolocation_silver.count():,}")

In [0]:
# ============================================================
# VERIFICATION
# ============================================================
print("=" * 60)
print("SILVER.GEOLOCATION VERIFICATION")
print("=" * 60)

df_silver = spark.table(f"{SILVER_DB}.geolocation")
bronze_count = spark.table(f"{BRONZE_DB}.geolocation").count()
silver_count = df_silver.count()

print(f"Bronze rows : {bronze_count:,}")
print(f"Silver rows : {silver_count:,}")
print(f"Reduction   : {(1-silver_count/bronze_count)*100:.1f}%")

print(f"\nTotal columns : {len(df_silver.columns)}")
for c in df_silver.columns:
    print(f"  {c}")

print("\nSample data:")
df_silver.show(5, truncate=False)

print("=" * 60)
print("silver.geolocation verification complete!")
print("=" * 60)